# 02 — Silver | CAT 2025 (SINAN / acidentes de trabalho)

**Fonte:** `laest.bronze.acgrbr25_raw` (decodificada no notebook `01_bronze_acgrbr25`)
**Saída:** `laest.silver.cat_2025` — mesmo número de registros da bronze (transformação linha a linha, sem agregação)

**O que esta camada faz:**

* **Tradução de códigos:** sexo, raça/cor, tipo de acidente, regime de tratamento e nome da UF (padrões SINAN/CID-10/IBGE)
* **Campos derivados:** `idade_anos` (decodificação da idade SINAN), `minutos_apos_inicio_jornada` (item 51 do dicionário — tempo transcorrido desde o início do expediente) e flag `trabalhador_plataforma` (CBO 5191-05 ciclista mensageiro, 5191-10 motofretista, 7823-15 motorista de aplicativo)
* **Sanitização:** `ano_nascimento` fora de 1915–2011 e `data_acidente` implausível (< 1990 ou posterior à notificação) viram nulos — erros de digitação; os valores originais permanecem intactos na bronze

**Pendências de tradução** — códigos mantidos com sufixo `_cod`, aguardando conferência com o dicionário CAT oficial: escolaridade, situação no mercado de trabalho, evolução do caso, emissão da CAT, local do acidente, terceirização e unidade do tempo na função.

**Autor/Data:** Vitor Silva — 03/11/2026

In [0]:
# Silver da CAT 2025: tradução de códigos, sanitização e campos derivados
from pyspark.sql import functions as F

BRONZE = 'laest.bronze.acgrbr25_raw'
SILVER = 'laest.silver.cat_2025'

# ---- Dicionários de tradução (padrões SINAN/CID-10/IBGE) ----
UF_NOME = {'11': 'Rondônia', '12': 'Acre', '13': 'Amazonas', '14': 'Roraima', '15': 'Pará',
           '16': 'Amapá', '17': 'Tocantins', '21': 'Maranhão', '22': 'Piauí', '23': 'Ceará',
           '24': 'Rio Grande do Norte', '25': 'Paraíba', '26': 'Pernambuco', '27': 'Alagoas',
           '28': 'Sergipe', '29': 'Bahia', '31': 'Minas Gerais', '32': 'Espírito Santo',
           '33': 'Rio de Janeiro', '35': 'São Paulo', '41': 'Paraná', '42': 'Santa Catarina',
           '43': 'Rio Grande do Sul', '50': 'Mato Grosso do Sul', '51': 'Mato Grosso',
           '52': 'Goiás', '53': 'Distrito Federal'}
SEXO = {'M': 'Masculino', 'F': 'Feminino', 'I': 'Ignorado'}
RACA_COR = {'1': 'Branca', '2': 'Preta', '3': 'Amarela', '4': 'Parda', '5': 'Indígena', '9': 'Ignorado'}
TIPO_ACIDENTE = {'1': 'Típico', '2': 'Trajeto', '9': 'Ignorado'}
REGIME_TRATAMENTO = {'1': 'Hospitalar', '2': 'Ambulatorial', '3': 'Ambos', '9': 'Ignorado'}
CBO_PLATAFORMA = ['519105', '519110', '782315']  # ciclista mensageiro, motofretista, motorista de aplicativo

def traduz(colname, dicionario, padrao='Não informado'):
    """Mapeia um código para seu rótulo; códigos fora do dicionário caem no padrão."""
    c = F.col(colname)
    pares = list(dicionario.items())
    expr = F.when(c == pares[0][0], F.lit(pares[0][1]))
    for chave, rotulo in pares[1:]:
        expr = expr.when(c == chave, F.lit(rotulo))
    return expr.otherwise(F.lit(padrao))

b = spark.table(BRONZE)

s = b.select(
    # --- notificação ---
    F.col('TP_NOT').alias('tp_notificacao'),
    F.col('ID_AGRAVO').alias('agravo'),
    F.col('DT_NOTIFIC').alias('data_notificacao'),
    F.col('NU_ANO').cast('int').alias('ano'),
    F.col('SG_UF_NOT').alias('uf_notificacao'),
    F.col('ID_MUNICIP').alias('municipio_notificacao'),
    # --- perfil do trabalhador ---
    traduz('CS_SEXO', SEXO).alias('sexo'),
    traduz('CS_RACA', RACA_COR).alias('raca_cor'),
    F.col('CS_ESCOL_N').alias('escolaridade_cod'),          # pendente: dicionário CAT
    F.when(F.col('ANO_NASC').cast('int').between(1915, 2011),
           F.col('ANO_NASC').cast('int')).alias('ano_nascimento'),
    F.when(F.col('NU_IDADE_N').between(4000, 4999), F.col('NU_IDADE_N') - 4000)
     .when(F.col('NU_IDADE_N').between(3000, 3999), F.round((F.col('NU_IDADE_N') - 3000) / 12))
     .alias('idade_anos'),                                 # idade SINAN: 4xxx=anos, 3xxx=meses
    F.col('SG_UF').alias('uf_residencia'),
    traduz('SG_UF', UF_NOME).alias('uf_residencia_nome'),
    F.col('ID_MN_RESI').alias('municipio_residencia'),
    F.col('ID_OCUPA_N').alias('ocupacao_cbo'),
    F.col('ID_OCUPA_N').isin(CBO_PLATAFORMA).alias('trabalhador_plataforma'),
    F.col('SIT_TRAB').alias('situacao_trabalho_cod'),       # pendente: dicionário CAT
    # --- vínculo / empresa ---
    F.col('CNAE').alias('cnae'),
    F.col('CNAE_PRIN').alias('cnae_principal'),
    F.col('UF_EMP').alias('uf_empregador'),
    F.col('MUN_EMP').alias('municipio_empregador'),
    F.col('TERCEIRIZA').alias('terceirizada_cod'),          # pendente: dicionário CAT
    # --- jornada e acidente ---
    F.col('NUTEMPO').alias('tempo_funcao_valor'),
    F.col('TPTEMPO').alias('tempo_funcao_unidade_cod'),    # pendente: dicionário CAT
    F.col('LOCAL_ACID').alias('local_acidente_cod'),        # pendente: dicionário CAT
    F.when((F.col('DT_ACID') >= F.lit('1990-01-01')) &
           (F.col('DT_ACID') <= F.col('DT_NOTIFIC')),
           F.col('DT_ACID')).alias('data_acidente'),
    F.col('HORA_ACID').cast('int').alias('hora_acidente'),
    F.col('MIN_ACID').cast('int').alias('minuto_acidente'),
    F.col('HORA_JOR').cast('int').alias('hora_inicio_jornada'),
    F.col('MIN_JOR').cast('int').alias('minuto_inicio_jornada'),
    # item 51 do dicionário: minutos transcorridos desde o início da jornada
    # (% 1440 cobre jornadas que cruzam a meia-noite)
    F.when(F.col('HORA_ACID').isNotNull() & F.col('HORA_JOR').isNotNull(),
           ((F.col('HORA_ACID').cast('int') * 60 + F.col('MIN_ACID').cast('int')) -
            (F.col('HORA_JOR').cast('int') * 60 + F.col('MIN_JOR').cast('int'))) % 1440
    ).alias('minutos_apos_inicio_jornada'),
    traduz('TIPO_ACID', TIPO_ACIDENTE).alias('tipo_acidente'),
    F.col('UF_ACID').alias('uf_acidente'),
    traduz('UF_ACID', UF_NOME).alias('uf_acidente_nome'),
    F.col('MUN_ACID').alias('municipio_acidente'),
    F.col('CID_ACID').alias('causa_acidente_cid10'),
    F.col('PART_CORP1').alias('parte_corpo_1'),
    F.col('PART_CORP2').alias('parte_corpo_2'),
    F.col('PART_CORP3').alias('parte_corpo_3'),
    F.col('CID_LESAO').alias('lesao_cid10'),
    # --- desfecho ---
    traduz('REGIME', REGIME_TRATAMENTO).alias('regime_tratamento'),
    F.col('EVOLUCAO').alias('evolucao_cod'),                # pendente: dicionário CAT
    F.col('DT_OBITO').alias('data_obito'),
    F.col('CAT').alias('cat_emitida_cod'),                  # pendente: dicionário CAT
    # --- rastreio ---
    F.col('_source_file'),
    F.col('_ingested_at'),
)

s.write.mode('overwrite').format('delta').saveAsTable(SILVER)

spark.sql(f"""COMMENT ON TABLE {SILVER} IS
'Camada silver da CAT 2025 (SINAN, acidentes de trabalho, agravo Y96). Traduz sexo, raca/cor,
tipo de acidente, regime de tratamento e nome das UFs; deriva idade_anos, minutos apos inicio da
jornada e flag trabalhador_plataforma (CBO 519105/519110/782315); sanitiza ano_nascimento e
data_acidente. Campos com sufixo _cod aguardam tradução pelo dicionário CAT oficial. Fonte: laest.bronze.acgrbr25_raw.'""")

print(f'{s.count():,} registros gravados em {SILVER}')

In [0]:
# Verificação: contagem (deve bater com a bronze: 646.427) e os primeiros indicadores
from pyspark.sql import functions as F

t = spark.table('laest.silver.cat_2025')
TOTAL = t.count()
print(f'Total silver: {TOTAL:,} (bronze: 646.427)')

def pct_por(coluna):
    t.groupBy(coluna) \
     .agg(F.count('*').alias('n'),
          F.round(100 * F.count('*') / TOTAL, 1).alias('pct')) \
     .orderBy(F.desc('n')).show(10, False)

print('=== Acidentes por sexo (%) ===');      pct_por('sexo')
print('=== Acidentes por raça/cor (%) ===');   pct_por('raca_cor')
print('=== Tipo de acidente (%) ===');         pct_por('tipo_acidente')
print('=== Trabalhador de plataforma (%) ==='); pct_por('trabalhador_plataforma')
print('=== Top 10 UFs do acidente (%) ===')
t.groupBy('uf_acidente_nome') \
 .agg(F.count('*').alias('n'), F.round(100 * F.count('*') / TOTAL, 1).alias('pct')) \
 .orderBy(F.desc('n')).show(10, False)

print('=== Óbitos ===')
t.select(F.count_if(F.col('data_obito').isNotNull()).alias('obitos'),
         F.round(100 * F.count_if(F.col('data_obito').isNotNull()) / TOTAL, 2).alias('pct')).show(1, False)